# 03 — Vibe Architect Studio: Napkin Sketch → Real BIM → Photoreal Render

Built for **architects, designers, and vibe coders** — run all cells once to launch a shareable **Gradio Studio App** (works on your laptop or iPad!).

### What Happens Under the Hood
1. **Drop a napkin sketch / Procreate floor plan OR type a vibe prompt**
2. **Gemma 4 31B** (optionally blended with your community **Weight-Space Trade LoRAs**) writes a parametric `@permitzip/bim-script` building model and silently self-repairs against `bim-lint`
3. **MutinyBIM** compiles the real building and exports:
   * Crisp **Architectural Floor Plan SVGs** (`sceneToSvg`) & **Vector PDF Sheets** (`bim-sheets`)
   * Standards-compliant **IFC 4.3** (`exportIfc`)
   * Interactive **3D GLB** (`IfcOpenShell` + `trimesh`)
4. **ControlNet Depth (`SDXL-Depth` or `FLUX.1-Depth`)** renders a **geometry-locked photorealistic image** from inside the 3D BIM model — zero hallucinated walls or floating doors!
5. Click **👁️ Critique & Redraw** to let Gemma 4's native vision inspect its own rendered floor plan + 3D cutaway and redesign it automatically.

### Hardware & Credit Budget
* **L4 (24GB VRAM, ~1.6 CCUs/hr):** Sweet spot — runs 4-bit Gemma 4 + CPU-offloaded SDXL-Depth ControlNet cleanly.
* **A100 (40GB / 80GB):** Faster generation & optional `FLUX.1-Depth-dev`.


In [ ]:
#@title 1. Check GPU, Clone Repos, Install Node 22, and Boot Warm MutinyBIM Export Server
!nvidia-smi

import json, subprocess, sys, time
from pathlib import Path

MIXER_REPO_URL = "https://github.com/5hyycjb77y/mutinybim-mixer.git" #@param {type:"string"}
WORK_DIR = Path("/content/work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
%cd /content/work

if not (WORK_DIR / "mutinybim-mixer").exists():
    !git clone --depth 1 {MIXER_REPO_URL} mutinybim-mixer || true

MIXER_DIR = WORK_DIR / "mutinybim-mixer"
assert MIXER_DIR.exists(), "Upload mutinybim-mixer to /content/work/mutinybim-mixer or set MIXER_REPO_URL."

SPEC = json.loads((MIXER_DIR / "spec.json").read_text())
REGISTRY = json.loads((MIXER_DIR / "registry.json").read_text())
MUTINY_DIR = WORK_DIR / "mutinybim"

if not MUTINY_DIR.exists():
    !git clone {SPEC["mutinybim"]["repo"]} {MUTINY_DIR}
!git -C {MUTINY_DIR} fetch --depth 1 origin {SPEC["mutinybim"]["commit"]} && git -C {MUTINY_DIR} checkout {SPEC["mutinybim"]["commit"]}

node_v = subprocess.run(["node", "-v"], capture_output=True, text=True)
if node_v.returncode != 0 or int(node_v.stdout.strip().lstrip("v").split(".")[0]) < 22:
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs
!node -v

if not (MUTINY_DIR / "node_modules").exists():
    !npm --prefix {MUTINY_DIR} install --no-audit --no-fund

verifier_log = open("/content/verifier.log", "w")
verifier_proc = subprocess.Popen(
    ["node", str(MIXER_DIR / "verifier/server.mjs"), "--mutinybim", str(MUTINY_DIR), "--port", "8787"],
    stdout=verifier_log,
    stderr=subprocess.STDOUT,
)

sys.path.insert(0, str(MIXER_DIR))
from verifier.verify_client import MutinyVerifierClient

client = MutinyVerifierClient("http://127.0.0.1:8787", timeout=60.0)
for _ in range(30):
    try:
        print("MutinyBIM Verifier & Exporter ready:", client.health())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Verifier failed to start — inspect /content/verifier.log")


In [ ]:
#@title 2. Install Studio Python Dependencies (Gradio, Unsloth, Diffusers, IfcOpenShell, Trimesh, CairoSVG)
!pip install -q --upgrade pip
!pip install -q gradio unsloth diffusers transformers accelerate peft safetensors ifcopenshell trimesh cairosvg pillow huggingface_hub


In [ ]:
#@title 3. Studio Configuration
RENDER_BACKEND = "sdxl_depth" #@param ["sdxl_depth", "flux_depth"]
ENABLE_VISION_SKETCH = True #@param {type:"boolean"}
ENABLE_THINKING = True #@param {type:"boolean"}
COMMUNITY_LORA_DIR = "" #@param {type:"string"}
RENDER_STEPS = 24 #@param {type:"integer"}
SHARE_PUBLIC_GRADIO_LINK = False #@param {type:"boolean"}

STUDIO_OUT = Path("/content/studio_runs")
STUDIO_OUT.mkdir(parents=True, exist_ok=True)
print(f"Studio ready | backend={RENDER_BACKEND} | thinking={ENABLE_THINKING} | custom_lora={COMMUNITY_LORA_DIR or 'base Gemma 4 31B'}")


In [ ]:
#@title 4. Load Gemma 4 31B (+ Optional Community Trade LoRA) & Define Self-Repair Generator
import torch
from unsloth import FastModel
from peft import PeftModel
from mixer.prompts import SYSTEM_PROMPT_TEMPLATE
from verifier.verify_client import extract_typescript

REF_BUNDLE = client.reference(compact=True)
KIT_REF_MD = REF_BUNDLE["markdown"]

base_model, tokenizer = FastModel.from_pretrained(
    model_name=SPEC["base_model"]["unsloth_id"],
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

if COMMUNITY_LORA_DIR.strip():
    model = PeftModel.from_pretrained(base_model, COMMUNITY_LORA_DIR.strip(), adapter_name="studio_active")
    print("Attached community LoRA:", COMMUNITY_LORA_DIR)
else:
    model = base_model
    print("Running base unsloth/gemma-4-31B-it (works out of the box!).")

FastModel.for_inference(model)


def _gemma_generate(messages, images=None, max_new_tokens=1800, temperature=0.7):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=ENABLE_THINKING,
        )
    except TypeError:
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    if images:
        try:
            inputs = tokenizer(text=[prompt_text], images=images, return_tensors="pt").to(model.device)
        except Exception:
            inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    else:
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    gen_cfg = SPEC["base_model"]["generation_defaults"]
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=gen_cfg["top_p"],
            top_k=gen_cfg["top_k"],
            do_sample=True,
        )
    return tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)


def apply_studio_faders(spatial_rigor: float, mep_detail: float, clash_polish: float) -> str:
    # Blend community adapters in weight space if registry.json / COMMUNITY_LORA_DIR adapters are loaded.
    adapters = REGISTRY.get("adapters", [])
    if not adapters:
        return f"Base Gemma 4 active (Faders noted: Spatial={spatial_rigor:.1f}, MEP={mep_detail:.1f}, ClashNeg={clash_polish:.1f})"
    try:
        from mixer.lora_ops import load_lora_deltas, merge_deltas, refactorize_deltas_to_lora, save_merged_lora
        deltas_list, weights, cfgs = [], [], []
        for entry in adapters:
            d, cfg = load_lora_deltas(entry["hf_repo"], revision=entry.get("revision"))
            trade = entry["trade"]
            if trade in ("architecture", "structural"):
                w = spatial_rigor
            elif trade in ("mechanical", "plumbing", "electrical"):
                w = mep_detail
            elif trade == "clash_negation":
                w = -abs(clash_polish)
            else:
                w = float(entry.get("default_weight", 0.7))
            deltas_list.append(d)
            weights.append(w)
            cfgs.append(cfg)
        merged = merge_deltas(deltas_list, weights, method="dare_ties", density=0.5, drop_rate=0.2)
        lora_sd, _ = refactorize_deltas_to_lora(merged, out_rank=16, lora_alpha=16)
        blend_dir = save_merged_lora(lora_sd, cfgs[0], STUDIO_OUT / "live_blend")
        if hasattr(model, "load_adapter"):
            if "studio_blend" in getattr(model, "peft_config", {}):
                model.delete_adapter("studio_blend")
            model.load_adapter(str(blend_dir), adapter_name="studio_blend")
            model.set_adapter("studio_blend")
        return f"Weight-space blended {len(adapters)} trade LoRAs!"
    except Exception as exc:
        return f"Fader blend fallback ({exc})"


def generate_and_export_bim(vibe_prompt: str, sketch_image=None, critique_context: str = "", critique_images=None):
    sys_msg = (
        SYSTEM_PROMPT_TEMPLATE
        + "\nAlways place at least one plan sheet (`documents.sheet` + `documents.planView` + `documents.place`) "
        + "so architectural drawings render cleanly.\n\n## MutinyBIM Script Kit Reference\n"
        + KIT_REF_MD[:4800]
    )

    user_parts = []
    images = []
    if sketch_image is not None and ENABLE_VISION_SKETCH:
        images.append(sketch_image)
        user_parts.append({"type": "image", "image": sketch_image})
    if critique_images:
        for img in critique_images:
            images.append(img)
            user_parts.append({"type": "image", "image": img})

    text_prompt = (
        f"Design Request:\n{vibe_prompt}\n"
        + (f"\nArt Director Critique & Improvement Instructions:\n{critique_context}\n" if critique_context else "")
        + "\nReturn one complete runnable ```ts block."
    )
    if images:
        user_parts.append({"type": "text", "text": text_prompt})
        user_content = user_parts
    else:
        user_content = text_prompt

    messages = [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_content},
    ]

    bundle = None
    ts_source = ""
    for attempt in range(2):
        raw_out = _gemma_generate(messages, images=images if attempt == 0 else None)
        ts_source = extract_typescript(raw_out)
        bundle = client.export_bundle(ts_source, project_name="Vibe Architect Studio", start="empty")
        if bundle.verify.ok and bundle.ifc_text:
            break
        if attempt == 0:
            err_msg = bundle.verify.error or bundle.verify.stage_failed or "Script failed verification."
            messages.append({"role": "assistant", "content": f"```ts\n{ts_source}\n```"})
            messages.append({
                "role": "user",
                "content": f"MutinyBIM verifier reported ({bundle.verify.stage_failed}): {err_msg}. Fix the TypeScript script and output one clean ```ts block.",
            })

    return ts_source, bundle


In [ ]:
#@title 5. End-to-End Studio Pipeline + Vision Critique & Redraw Loop
import base64
import time
from PIL import Image
from mixer.studio_render import (
    ART_DIRECTION_PRESETS,
    CAMERA_PRESETS,
    VIBE_INSPIRATION_CHIPS,
    ifc_to_glb_and_control,
    render_photoreal,
    svg_to_png,
)


def _format_sheets_html(sheets: list[dict]) -> str:
    if not sheets:
        return "<div style='padding:24px;color:#666;'>No sheets were emitted by this script. Try asking for an architectural plan sheet (A-101).</div>"
    cards = []
    for sh in sheets:
        cards.append(
            f"<div style='margin-bottom:20px;border:1px solid #ddd;border-radius:10px;padding:12px;background:#fff;'>"
            f"<div style='font-weight:600;margin-bottom:8px;color:#111;'>Sheet {sh['number']} — {sh['title']}</div>"
            f"<div style='width:100%;overflow:auto;background:#fafafa;border:1px solid #eee;'>{sh['svg']}</div>"
            f"</div>"
        )
    return "\n".join(cards)


def run_studio_pipeline(
    vibe_prompt: str,
    sketch_image,
    style_key: str,
    camera_key: str,
    spatial_rigor: float,
    mep_detail: float,
    clash_polish: float,
    seed: int = 42,
):
    run_id = f"run_{int(time.time())}"
    run_dir = STUDIO_OUT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    fader_msg = apply_studio_faders(spatial_rigor, mep_detail, clash_polish)
    ts_source, bundle = generate_and_export_bim(vibe_prompt, sketch_image=sketch_image)

    ts_path = run_dir / "model.ts"
    ts_path.write_text(ts_source, encoding="utf-8")

    ifc_path = None
    pdf_path = None
    glb_path = None
    depth_path = None
    clay_path = None
    photoreal_path = None

    if bundle.ifc_text:
        ifc_path = str(run_dir / "model.ifc")
        Path(ifc_path).write_text(bundle.ifc_text, encoding="utf-8")
        geom = ifc_to_glb_and_control(ifc_path, run_dir, camera_preset=camera_key)
        glb_path = geom["glb_path"]
        depth_path = geom["depth_path"]
        clay_path = geom["clay_path"]
        if depth_path:
            photoreal_path = render_photoreal(
                depth_path,
                vibe_prompt,
                style_key,
                camera_key,
                run_dir / "render.png",
                backend=RENDER_BACKEND,
                steps=RENDER_STEPS,
                seed=int(seed),
            )

    if bundle.pdf_base64:
        pdf_path = str(run_dir / "sheets.pdf")
        Path(pdf_path).write_bytes(base64.b64decode(bundle.pdf_base64))

    total_elems = sum(bundle.verify.kinds.values())
    clashes = int(bundle.verify.lint_summary.get("errors", 0))
    warnings = int(bundle.verify.lint_summary.get("warnings", 0))
    badge = "✅" if bundle.verify.ok else "⚠️"
    status_md = (
        f"### {badge} Clash-Free Score: `{bundle.verify.reward:.2f}` · "
        f"`{clashes}` Errors / `{warnings}` Warnings · `{total_elems}` BIM Elements · "
        f"`{len(bundle.sheets)}` Sheets\n"
        f"*{fader_msg}* · Kinds: `{json.dumps(bundle.verify.kinds)}`"
    )
    if bundle.verify.error:
        status_md += f"\n\n> Note: `{bundle.verify.error}`"

    sheets_html = _format_sheets_html(bundle.sheets)
    downloads = [p for p in (ifc_path, pdf_path, glb_path, photoreal_path, str(ts_path)) if p]
    state = {
        "vibe_prompt": vibe_prompt,
        "ts_source": ts_source,
        "clay_path": clay_path,
        "first_svg": bundle.sheets[0]["svg"] if bundle.sheets else "",
        "run_dir": str(run_dir),
    }

    return (
        status_md,
        photoreal_path,
        clay_path,
        glb_path,
        sheets_html,
        downloads,
        ts_source,
        state,
    )


def critique_and_redraw(
    state: dict,
    sketch_image,
    style_key: str,
    camera_key: str,
    spatial_rigor: float,
    mep_detail: float,
    clash_polish: float,
    seed: int = 42,
):
    if not state or not state.get("ts_source"):
        return ("Run a generation first before critiquing!", None, None, None, "", [], "", state)

    run_dir = Path(state["run_dir"])
    critique_imgs = []
    if state.get("first_svg"):
        plan_png = svg_to_png(state["first_svg"], run_dir / "plan_critique.png")
        critique_imgs.append(Image.open(plan_png).convert("RGB"))
    if state.get("clay_path") and Path(state["clay_path"]).exists():
        critique_imgs.append(Image.open(state["clay_path"]).convert("RGB"))

    critique_prompt = (
        "Inspect the architectural floor plan and 3D cutaway of your previous MutinyBIM design. "
        "Identify any spatial circulation bottlenecks, awkward door placements, missing daylight windows, "
        "or cramped zones, then immediately output an improved TypeScript script.\n\n"
        f"Previous script:\n```ts\n{state['ts_source']}\n```"
    )
    ts_source, bundle = generate_and_export_bim(
        state["vibe_prompt"],
        sketch_image=sketch_image,
        critique_context=critique_prompt,
        critique_images=critique_imgs,
    )
    state["ts_source"] = ts_source

    new_dir = STUDIO_OUT / f"critique_{int(time.time())}"
    new_dir.mkdir(parents=True, exist_ok=True)
    ts_path = new_dir / "model.ts"
    ts_path.write_text(ts_source, encoding="utf-8")

    ifc_path = pdf_path = glb_path = clay_path = photoreal_path = None
    if bundle.ifc_text:
        ifc_path = str(new_dir / "model.ifc")
        Path(ifc_path).write_text(bundle.ifc_text, encoding="utf-8")
        geom = ifc_to_glb_and_control(ifc_path, new_dir, camera_preset=camera_key)
        glb_path, depth_path, clay_path = geom["glb_path"], geom["depth_path"], geom["clay_path"]
        if depth_path:
            photoreal_path = render_photoreal(
                depth_path, state["vibe_prompt"], style_key, camera_key, new_dir / "render.png",
                backend=RENDER_BACKEND, steps=RENDER_STEPS, seed=int(seed) + 1,
            )

    if bundle.pdf_base64:
        pdf_path = str(new_dir / "sheets.pdf")
        Path(pdf_path).write_bytes(base64.b64decode(bundle.pdf_base64))

    total_elems = sum(bundle.verify.kinds.values())
    clashes = int(bundle.verify.lint_summary.get("errors", 0))
    status_md = (
        f"### 👁️ Vision Critique Redraw Complete · Reward `{bundle.verify.reward:.2f}` · "
        f"`{clashes}` Errors · `{total_elems}` Elements"
    )
    downloads = [p for p in (ifc_path, pdf_path, glb_path, photoreal_path, str(ts_path)) if p]
    state.update({
        "clay_path": clay_path,
        "first_svg": bundle.sheets[0]["svg"] if bundle.sheets else "",
        "run_dir": str(new_dir),
    })
    return (
        status_md,
        photoreal_path,
        clay_path,
        glb_path,
        _format_sheets_html(bundle.sheets),
        downloads,
        ts_source,
        state,
    )


In [ ]:
#@title 6. Launch Vibe Architect Studio (Gradio Web UI)
import gradio as gr

with gr.Blocks(title="MutinyBIM Vibe Architect Studio", theme=gr.themes.Soft()) as studio:
    gr.Markdown(
        "# 🏗️ MutinyBIM Vibe Architect Studio\n"
        "Drop a napkin sketch or describe a space → get a **real parametric IFC 4.3 building**, "
        "**printable architectural sheets**, **interactive 3D model**, and a **geometry-locked photoreal render**."
    )
    studio_state = gr.State({})

    with gr.Row():
        with gr.Column(scale=4):
            sketch_in = gr.Image(label="Napkin Sketch / Floor Plan Screenshot (Optional)", type="pil", height=220)
            vibe_in = gr.Textbox(
                label="Vibe Prompt",
                lines=4,
                value=VIBE_INSPIRATION_CHIPS[0]["prompt"],
                placeholder="Describe your space, materials, rooms, daylighting, or MEP vibe...",
            )
            with gr.Row():
                for chip in VIBE_INSPIRATION_CHIPS:
                    btn = gr.Button(chip["title"], size="sm")
                    btn.click(fn=lambda p=chip["prompt"]: p, outputs=vibe_in)

            style_in = gr.Dropdown(
                label="Art Direction",
                choices=list(ART_DIRECTION_PRESETS.keys()),
                value="ArchDaily Golden Hour Timber",
            )
            camera_in = gr.Radio(
                label="Camera Angle",
                choices=[(v["label"], k) for k, v in CAMERA_PRESETS.items()],
                value="isometric_cutaway",
            )
            seed_in = gr.Number(label="Render Seed", value=42, precision=0)

            with gr.Accordion("🎛️ Design Mixing Desk (Weight-Space Faders)", open=False):
                spatial_slider = gr.Slider(0.0, 1.5, value=0.9, step=0.1, label="Spatial & Architectural Rigor")
                mep_slider = gr.Slider(0.0, 1.5, value=0.7, step=0.1, label="MEP & Code Engineering Detail")
                clash_slider = gr.Slider(0.0, 1.0, value=0.3, step=0.1, label="Clash-Negation Polish (-λ)")

            with gr.Row():
                gen_btn = gr.Button("✨ Generate Real Building", variant="primary", scale=2)
                critique_btn = gr.Button("👁️ Critique & Redraw", variant="secondary", scale=1)

        with gr.Column(scale=6):
            status_out = gr.Markdown("Ready — click **✨ Generate Real Building** to start.")
            with gr.Tabs():
                with gr.Tab("🎨 Photoreal Render"):
                    photoreal_out = gr.Image(label="Geometry-Locked Photoreal Render", type="filepath")
                    clay_out = gr.Image(label="True BIM 3D Clay / Geometry Reference", type="filepath")
                with gr.Tab("🧊 Interactive 3D Model"):
                    glb_out = gr.Model3D(label="Interactive IFC → GLB Model", height=520)
                with gr.Tab("📐 Architectural Plan Sheets"):
                    sheets_out = gr.HTML(label="MutinyBIM Vector SVG Sheets")
                with gr.Tab("📦 Deliverables & Code"):
                    files_out = gr.Files(label="Download IFC 4.3, Vector PDF, 3D GLB & Script")
                    code_out = gr.Code(label="Generated @permitzip/bim-script", language="typescript")

    inputs = [vibe_in, sketch_in, style_in, camera_in, spatial_slider, mep_slider, clash_slider, seed_in]
    outputs = [status_out, photoreal_out, clay_out, glb_out, sheets_out, files_out, code_out, studio_state]

    gen_btn.click(fn=run_studio_pipeline, inputs=inputs, outputs=outputs)
    critique_btn.click(
        fn=critique_and_redraw,
        inputs=[studio_state, sketch_in, style_in, camera_in, spatial_slider, mep_slider, clash_slider, seed_in],
        outputs=outputs,
    )

studio.queue().launch(share=SHARE_PUBLIC_GRADIO_LINK, debug=False)
